# Time-Binned Geomagnetic Activity Dataset

**One row = one fixed time bin (3 hours).** Instead of anchoring on CME events, we put every dataset on a single regular clock and frame the problem as time-series forecasting:

> *Given the recent history of solar-wind conditions, what will the geomagnetic activity (ap) be N hours from now?*

**Datasets used**
- `omni_solar_wind_parameters` — hourly solar wind at L1; the source of both features (Bz, speed, density, pressure, IMF) **and** the target (`ap_index_nt`).
- `solar_flare_events` — collapsed to per-bin counts + max GOES class.
- `space_weather_events` (CME rows) — collapsed to per-bin counts + max speed.

**Leakage guardrails baked in (see each step):**
1. Rolling features are **trailing only** (`closed='right'`, never `center=True`).
2. The target is built with a forward `shift(-HORIZON)`; features never reference a bin ≥ the target's time.
3. Train/test split is **temporal** (earlier → train, later → test); no shuffling.
4. Gaps are **forward-filled only** (never back-filled or interpolated forward).
5. A **persistence baseline** is included — the real bar to beat, not the mean.
6. Scaling/imputation must be fit on **train only** (done in the modeling pipeline).

In [1]:
import pandas as pd
import numpy as np
from datasets import load_dataset

ds_omni   = load_dataset("juliensimon/omni-solar-wind-parameters", split="all")
ds_flares = load_dataset("juliensimon/solar-flare-events", split="all")
ds_events = load_dataset("juliensimon/donki-space-weather-events", split="all")

df_omni   = ds_omni.to_pandas()
df_flares = ds_flares.to_pandas()
df_events = ds_events.to_pandas()

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

- `BIN` = 3h grid (matches how Kp/ap are reported).
- `WINDOW_BINS` = how many trailing bins go into each feature window (8 bins = 24h of history).
- `HORIZON_BINS` = how far ahead we forecast (8 bins = 24h ahead). Change this to make it a 3h / 6h / 24h forecast.
- Flares start 2017-02, so the flare features are only meaningful from 2017 on. We keep the full range; flare columns are 0 before then (no flare observed).

In [2]:
BIN          = '3h'
WINDOW_BINS  = 8     # 8 * 3h = 24h trailing history per feature
HORIZON_BINS = 8     # forecast ap 24h ahead
GAP_FFILL_LIMIT = 4  # forward-fill at most 4 bins (12h) across a data gap

START = '2010-01-01'
END   = '2024-12-31'

WIND_COLS = ['bz_gsm_nt', 'b_magnitude_avg_nt', 'flow_speed_kms',
             'proton_density_cm3', 'flow_pressure_npa', 'electric_field_mvpm']
TARGET_COL = 'ap_index_nt'

## 1. Build the regular time grid + bin the solar wind

OMNI is hourly. We resample it onto the 3h grid by averaging the wind within each bin, and take the **max** ap in each bin (storm-relevant). `resample` produces a continuous, gap-aware grid — missing bins become NaN rather than silently vanishing.

In [3]:
df_omni['datetime'] = pd.to_datetime(df_omni['datetime'])
df_omni = (df_omni[(df_omni['datetime'] >= START) & (df_omni['datetime'] <= END)]
           .set_index('datetime')
           .sort_index())

# Within-bin aggregation: mean for wind features, max for the ap target.
grid = df_omni.resample(BIN).agg({**{c: 'mean' for c in WIND_COLS},
                                  TARGET_COL: 'max'})

print(f"Grid: {len(grid):,} bins  |  {grid.index.min()} -> {grid.index.max()}")
print(f"Missing-bin rate per column:\n{grid.isna().mean().round(3)}")
grid.head()

Grid: 43,825 bins  |  2010-01-01 00:00:00 -> 2024-12-31 00:00:00
Missing-bin rate per column:
bz_gsm_nt              0.002
b_magnitude_avg_nt     0.002
flow_speed_kms         0.004
proton_density_cm3     0.008
flow_pressure_npa      0.008
electric_field_mvpm    0.004
ap_index_nt            0.000
dtype: float64


,bz_gsm_nt,b_magnitude_avg_nt,flow_speed_kms,proton_density_cm3,flow_pressure_npa,electric_field_mvpm,ap_index_nt
datetime,,,,,,,
2010-01-01 00:00:00,1.466667,3.000000,281.000000,3.766667,0.516667,-0.413333,0.0
2010-01-01 03:00:00,0.866667,2.966667,281.000000,3.666667,0.530000,-0.240000,0.0
2010-01-01 06:00:00,0.433333,2.766667,286.666667,3.800000,0.573333,-0.123333,0.0
2010-01-01 09:00:00,1.066667,3.133333,290.000000,3.866667,0.620000,-0.306667,0.0
2010-01-01 12:00:00,0.966667,4.133333,290.000000,4.400000,0.716667,-0.280000,0.0


## 2. Bin the flares — counts + max class per bin

Flares collapse to two per-bin features: how many flares peaked in the bin, and the strongest GOES class. Joined on the bin timestamp (no fuzzy tolerance).

In [4]:
class_map = {'A': 1, 'B': 2, 'C': 3, 'M': 4, 'X': 5}

f = df_flares.copy()
f['peak_time'] = pd.to_datetime(f['peak_time'])
f = f.dropna(subset=['peak_time'])
f = f[(f['peak_time'] >= START) & (f['peak_time'] <= END)]
f['flare_class_num'] = f['goes_class_letter'].map(class_map)
f = f.set_index('peak_time').sort_index()

flare_binned = f.resample(BIN).agg(
    flare_count=('flare_class_num', 'size'),
    flare_max_class=('flare_class_num', 'max'),
)
# size counts rows even when class is NaN; recount only valid-class flares
flare_binned['flare_count'] = f['flare_class_num'].resample(BIN).count()
flare_binned = flare_binned.reindex(grid.index)
flare_binned['flare_count'] = flare_binned['flare_count'].fillna(0)
flare_binned['flare_max_class'] = flare_binned['flare_max_class'].fillna(0)

flare_binned.describe()

,flare_count,flare_max_class
count,43825.000000,43825.000000
mean,0.347108,0.544233
std,0.849852,1.161326
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.000000
max,7.000000,5.000000


## 3. Bin the CMEs — counts + max speed per bin

Same treatment for CME launches. Note these are *launch* features: as discussed, the CME's own Bz isn't knowable at launch, so these are weak background signals, not the main driver. They're included as context, not the anchor.

In [5]:
c = df_events[df_events['event_type'] == 'CME'].copy()
c['start_time'] = pd.to_datetime(c['start_time'], utc=True).dt.tz_localize(None)
c = c[(c['start_time'] >= START) & (c['start_time'] <= END)]
c = c.set_index('start_time').sort_index()

cme_binned = c.resample(BIN).agg(
    cme_count=('cme_speed_kms', 'size'),
    cme_max_speed=('cme_speed_kms', 'max'),
)
cme_binned = cme_binned.reindex(grid.index)
cme_binned['cme_count'] = cme_binned['cme_count'].fillna(0)
cme_binned['cme_max_speed'] = cme_binned['cme_max_speed'].fillna(0)

cme_binned.describe()

,cme_count,cme_max_speed
count,43825.000000,43825.000000
mean,0.166389,77.817691
std,0.431368,220.175013
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.000000
max,5.000000,3529.000000


## 4. Assemble the panel + forward-fill gaps (guardrail #4)

Join everything on the bin timestamp. Then **forward-fill only** the wind columns across short gaps — forward-fill uses past values, so it cannot leak the future. We cap the fill length so a long outage doesn't get smeared over (`GAP_FFILL_LIMIT`). We deliberately do **not** fill the target.

In [6]:
panel = grid.join(flare_binned).join(cme_binned)

# Forward-fill ONLY the continuous wind features, only across short gaps.
panel[WIND_COLS] = panel[WIND_COLS].ffill(limit=GAP_FFILL_LIMIT)

print(f"Panel shape: {panel.shape}")
print(f"Remaining NaN rate (wind):\n{panel[WIND_COLS].isna().mean().round(3)}")

Panel shape: (43825, 11)
Remaining NaN rate (wind):
bz_gsm_nt              0.001
b_magnitude_avg_nt     0.000
flow_speed_kms         0.001
proton_density_cm3     0.002
flow_pressure_npa      0.002
electric_field_mvpm    0.001
dtype: float64


## 5. Trailing-window features (guardrail #1)

For each wind column we compute rolling statistics over the last `WINDOW_BINS` bins. `rolling` is **trailing by default** (`closed='right'`, includes the current bin, never future ones). We add:
- `*_mean`, `*_min`, `*_max` over the window
- `*_last` = the current bin's value
- `*_trend` = current value minus the value at the start of the window (rate of change)

The `_now` snapshot of recent measured Bz / E-field is exactly the info that *is* predictive a few hours ahead — the whole reason this framing has skill where the CME-launch one doesn't.

In [7]:
feat = pd.DataFrame(index=panel.index)

for col in WIND_COLS:
    s = panel[col]
    roll = s.rolling(WINDOW_BINS, min_periods=WINDOW_BINS // 2)  # trailing
    feat[f'{col}_mean']  = roll.mean()
    feat[f'{col}_min']   = roll.min()
    feat[f'{col}_max']   = roll.max()
    feat[f'{col}_last']  = s                          # current bin value
    feat[f'{col}_trend'] = s - s.shift(WINDOW_BINS - 1)  # change across window

# Event features: count over the trailing window + current-bin max intensity
for col, agg_now in [('flare_count', 'flare_max_class'), ('cme_count', 'cme_max_speed')]:
    feat[f'{col}_win'] = panel[col].rolling(WINDOW_BINS, min_periods=1).sum()
feat['flare_max_class_now'] = panel['flare_max_class']
feat['cme_max_speed_now']   = panel['cme_max_speed']

# Persistence feature / baseline input: most recent observed ap
feat['ap_now'] = panel[TARGET_COL]

print(f"{feat.shape[1]} features built")
feat.head()

35 features built


,bz_gsm_nt_mean,bz_gsm_nt_min,bz_gsm_nt_max,bz_gsm_nt_last,bz_gsm_nt_trend,b_magnitude_avg_nt_mean,b_magnitude_avg_nt_min,b_magnitude_avg_nt_max,b_magnitude_avg_nt_last,b_magnitude_avg_nt_trend,...,electric_field_mvpm_mean,electric_field_mvpm_min,electric_field_mvpm_max,electric_field_mvpm_last,electric_field_mvpm_trend,flare_count_win,cme_count_win,flare_max_class_now,cme_max_speed_now,ap_now
datetime,,,,,,,,,,,,,,,,,,,,,
2010-01-01 00:00:00,NaN,NaN,NaN,1.466667,NaN,NaN,NaN,NaN,3.000000,NaN,...,NaN,NaN,NaN,-0.413333,NaN,0.0,0.0,0.0,0.0,0.0
2010-01-01 03:00:00,NaN,NaN,NaN,0.866667,NaN,NaN,NaN,NaN,2.966667,NaN,...,NaN,NaN,NaN,-0.240000,NaN,0.0,0.0,0.0,0.0,0.0
2010-01-01 06:00:00,NaN,NaN,NaN,0.433333,NaN,NaN,NaN,NaN,2.766667,NaN,...,NaN,NaN,NaN,-0.123333,NaN,0.0,0.0,0.0,0.0,0.0
2010-01-01 09:00:00,0.958333,0.433333,1.466667,1.066667,NaN,2.966667,2.766667,3.133333,3.133333,NaN,...,-0.270833,-0.413333,-0.123333,-0.306667,NaN,0.0,0.0,0.0,0.0,0.0
2010-01-01 12:00:00,0.960000,0.433333,1.466667,0.966667,NaN,3.200000,2.766667,4.133333,4.133333,NaN,...,-0.272667,-0.413333,-0.123333,-0.280000,NaN,0.0,0.0,0.0,0.0,0.0


## 6. Forward target (guardrail #2)

The label is ap `HORIZON_BINS` ahead, via `shift(-HORIZON_BINS)`. Every feature at row T uses only bins ≤ T; the target is strictly at T + horizon. We assemble features + target, then drop rows where either side is incomplete (window not yet full at the start, or no future ap at the end).

In [8]:
feat['ap_target'] = panel[TARGET_COL].shift(-HORIZON_BINS)

dataset = feat.dropna(subset=['ap_now', 'ap_target']).copy()

# Leakage self-check: the target's source bin must be strictly in the future.
assert HORIZON_BINS > 0, "target must look forward"
print(f"Final dataset: {dataset.shape[0]:,} rows x {dataset.shape[1]-1} features + target")
print(f"ap_target stats:\n{dataset['ap_target'].describe().round(1)}")
print(f"Storm bins (ap_target > 50): {(dataset['ap_target'] > 50).mean():.1%}")

Final dataset: 43,817 rows x 35 features + target
ap_target stats:
count    43817.0
mean         8.6
std         12.7
min          0.0
25%          3.0
50%          5.0
75%          9.0
max        400.0
Name: ap_target, dtype: float64
Storm bins (ap_target > 50): 1.3%


## 7. Temporal split + persistence baseline (guardrails #3, #5)

Split by time — first 80% of bins train, last 20% test. Then the baseline that actually matters: **persistence**, i.e. predict that ap `HORIZON` ahead equals ap now. Any real model has to beat this, not just the mean.

In [9]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

dataset = dataset.sort_index()
split = int(len(dataset) * 0.8)
train, test = dataset.iloc[:split], dataset.iloc[split:]
print(f"Train: {len(train):,}  ({train.index.min()} -> {train.index.max()})")
print(f"Test:  {len(test):,}  ({test.index.min()} -> {test.index.max()})")

def evaluate(name, y_true, y_pred):
    print(f"{name:<22} MAE={mean_absolute_error(y_true, y_pred):6.2f}  "
          f"RMSE={root_mean_squared_error(y_true, y_pred):7.2f}  "
          f"R2={r2_score(y_true, y_pred):.3f}")

y_test = test['ap_target']
evaluate('Mean baseline',  y_test, np.full(len(y_test), train['ap_target'].mean()))
evaluate('Persistence',    y_test, test['ap_now'])  # <-- the real bar

Train: 35,053  (2010-01-01 00:00:00 -> 2021-12-30 12:00:00)
Test:  8,764  (2021-12-30 15:00:00 -> 2024-12-30 00:00:00)
Mean baseline          MAE=  7.41  RMSE=  18.08  R2=-0.025
Persistence            MAE=  9.91  RMSE=  23.08  R2=-0.672


## 8. Save

Saved with the bin timestamp as the index so the temporal ordering is preserved for any downstream modeling. Remember when modeling: **fit the scaler/imputer on `train` only** (guardrail #6) — wrap them in a `Pipeline` and call `.fit` on the training rows alone, and use `TimeSeriesSplit` (not `KFold`) for any tuning.

In [ ]:
dataset.to_csv('../data/time_binned_dataset.csv')
print(f"Saved ../data/time_binned_dataset.csv  ({dataset.shape[0]:,} rows, {dataset.shape[1]} cols)")
print(f"\nForecast horizon: {HORIZON_BINS} bins = {HORIZON_BINS * 3}h ahead")
print(f"Feature history:  {WINDOW_BINS} bins = {WINDOW_BINS * 3}h trailing")